In [2]:
import yfinance as yf
import pandas as pd
import time
import requests
from requests.exceptions import HTTPError

# Configure custom session with browser-like headers
session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
})

# Stock parameters
ticker_symbol = "HINDALCO.NS"
start_date = "2007-01-01"
end_date = "2025-12-31"

def fetch_stock_data(max_retries=5, initial_delay=10):
    current_delay = initial_delay
    for attempt in range(max_retries):
        try:
            stock = yf.Ticker(ticker_symbol, session=session)
            data = stock.history(
                start=start_date,
                end=end_date,
                interval="1d",
                actions=False
            )
            
            if data.empty:
                raise ValueError("No data returned - check dates/ticker")
                
            processed_data = data[['Open', 'Close', 'Volume']].reset_index()
            processed_data['Date'] = pd.to_datetime(processed_data['Date']).dt.date
            return processed_data
            
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {str(e)}")
            if attempt < max_retries - 1:
                print(f"Waiting {current_delay} seconds...")
                time.sleep(current_delay)
                current_delay *= 2  # Exponential backoff
                
    print("All retries failed. Try again later.")
    return None

# Main execution
if __name__ == "__main__":
    print(f"Fetching {ticker_symbol} data from {start_date} to {end_date}")
    stock_data = fetch_stock_data()
    
    if stock_data is not None:
        print("\nFirst 5 rows:")
        print(stock_data.head())
        #stock_data.to_csv("HINDALCO.NS_Stock_Data.csv", index=False)
        #print("\nSaved to HINDALCO.NS_Stock_Data.csv")
        print(f"\nDate Range: {stock_data['Date'].min()} to {stock_data['Date'].max()}")
    else:
        print("\nFailed to fetch data. Possible solutions:")
        print("- Use a VPN/proxy")
        print("- Verify ticker at https://finance.yahoo.com/quote/BA.L")
        print("- Wait 1 hour and try again")

Fetching HINDALCO.NS data from 2007-01-01 to 2025-12-31

First 5 rows:
         Date        Open       Close   Volume
0  2007-01-02  136.324866  137.522049  1737983
1  2007-01-03  139.028174  138.410278  6568517
2  2007-01-04  138.255803  135.011810  5541476
3  2007-01-05  134.278078  132.154037  5296083
4  2007-01-08  131.381624  129.025879  5202449

Date Range: 2007-01-02 to 2025-02-27


In [3]:
import yfinance as yf
import pandas as pd
import time
import requests
import random
from requests.exceptions import HTTPError

# Configure custom session with browser-like headers
session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
})

# Ticker list (FTSE 100 companies)
TICKERS = [
            "HINDALCO.NS", "BAJAJ-AUTO.NS", "EICHERMOT.NS", "HEROMOTOCO.NS", "M&M.NS", "MARUTI.NS", "TATAMOTORS.NS", 
           "HDFCBANK.NS", "ICICIBANK.NS", "INDUSINDBK.NS", "KOTAKBANK.NS", "SBIN.NS", "AXISBANK.NS", "ULTRACEMCO.NS", 
           "BEL.NS", "BPCL.NS", "IOC.NS", "ONGC.NS", "RELIANCE.NS", "LT.NS", "BAJAJFINSV.NS", "BAJFINANCE.NS", 
           "SHRIRAMFIN.NS", "HINDUNILVR.NS", "TATACONSUM.NS", "ITC.NS", "NESTLEIND.NS", "HDFCLIFE.NS", "SBILIFE.NS", 
           "COALINDIA.NS", "ADANIENT.NS", "APOLLOHOSP.NS", "ADANIPORTS.NS", "ASIANPAINT.NS", "CIPLA.NS", "DRREDDY.NS", 
           "SUNPHARMA.NS", "NTPC.NS", "POWERGRID.NS", "TITAN.NS", "TRENT.NS", "HCLTECH.NS", "INFY.NS", "TCS.NS", 
           "TECHM.NS", "WIPRO.NS", "JSWSTEEL.NS", "TATASTEEL.NS", "BHARTIARTL.NS", "GRASIM.NS"
]

# Date parameters
START_DATE = "2007-01-01"
END_DATE = "2025-12-31"

def fetch_stock_data(ticker, max_retries=3, initial_delay=5):
    current_delay = initial_delay
    for attempt in range(max_retries + 1):
        try:
            stock = yf.Ticker(ticker, session=session)
            data = stock.history(
                start=START_DATE,
                end=END_DATE,
                interval="1d",
                actions=False
            )
            
            if data.empty:
                print(f"No data for {ticker}")
                return None
                
            processed_data = data[['Open', 'Close', 'Volume']].reset_index()
            processed_data['Date'] = pd.to_datetime(processed_data['Date']).dt.date
            processed_data['Ticker'] = ticker  # Add Ticker column
            return processed_data
            
        except Exception as e:
            if attempt < max_retries:
                sleep_time = current_delay + random.uniform(0, 3)
                print(f"Retry {attempt+1} for {ticker} in {sleep_time:.1f}s: {str(e)}")
                time.sleep(sleep_time)
                current_delay *= 2
            else:
                print(f"Failed {ticker} after {max_retries} retries")
                return None

def main():
    all_data = []
    total_tickers = len(TICKERS)
    success_count = 0
    
    for idx, ticker in enumerate(TICKERS, 1):
        print(f"\nProcessing {ticker} ({idx}/{total_tickers})")
        
        # Random delay to avoid detection
        time.sleep(random.uniform(0.5, 1.5))
        
        data = fetch_stock_data(ticker)
        
        if data is not None:
            all_data.append(data)
            success_count += 1
            
    if all_data:
        # Merge all DataFrames
        combined_df = pd.concat(all_data, ignore_index=True)
        
        # Save merged data
        combined_df.to_csv("NSEI_companies_stock.csv", index=False)
        print(f"\nSuccessfully saved {len(combined_df)} rows from {success_count} companies")
        print("Columns in final dataset:", combined_df.columns.tolist())
    else:
        print("\nNo data was collected")
        
    print(f"\nSuccess rate: {success_count}/{total_tickers} ({success_count/total_tickers:.1%})")

if __name__ == "__main__":
    start_time = time.time()
    main()
    print(f"\nTotal execution time: {(time.time() - start_time)/60:.2f} minutes")


Processing HINDALCO.NS (1/50)

Processing BAJAJ-AUTO.NS (2/50)

Processing EICHERMOT.NS (3/50)

Processing HEROMOTOCO.NS (4/50)

Processing M&M.NS (5/50)

Processing MARUTI.NS (6/50)

Processing TATAMOTORS.NS (7/50)

Processing HDFCBANK.NS (8/50)

Processing ICICIBANK.NS (9/50)

Processing INDUSINDBK.NS (10/50)

Processing KOTAKBANK.NS (11/50)

Processing SBIN.NS (12/50)

Processing AXISBANK.NS (13/50)

Processing ULTRACEMCO.NS (14/50)

Processing BEL.NS (15/50)

Processing BPCL.NS (16/50)

Processing IOC.NS (17/50)

Processing ONGC.NS (18/50)

Processing RELIANCE.NS (19/50)

Processing LT.NS (20/50)

Processing BAJAJFINSV.NS (21/50)

Processing BAJFINANCE.NS (22/50)

Processing SHRIRAMFIN.NS (23/50)

Processing HINDUNILVR.NS (24/50)

Processing TATACONSUM.NS (25/50)

Processing ITC.NS (26/50)

Processing NESTLEIND.NS (27/50)

Processing HDFCLIFE.NS (28/50)

Processing SBILIFE.NS (29/50)

Processing COALINDIA.NS (30/50)

Processing ADANIENT.NS (31/50)

Processing APOLLOHOSP.NS (32/50)